# Load Data and Chunks data

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Load Documents with hardcoded text
documents = [
    Document(
        page_content="""
            ## Before we begin

            > [!NOTE] If you're new to Svelte or SvelteKit we recommend checking out the [interactive tutorial](/tutorial/kit).
            >
            > If you get stuck, reach out for help in the [Discord chatroom](/chat).

            ## What is SvelteKit?

            SvelteKit is a framework for rapidly developing robust, performant web applications using [Svelte](../svelte). If you're coming from React, SvelteKit is similar to Next. If you're coming from Vue, SvelteKit is similar to Nuxt.

            To learn more about the kinds of applications you can build with SvelteKit, see the [documentation regarding project types](project-types).

            ## What is Svelte?

            In short, Svelte is a way of writing user interface components — like a navigation bar, comment section, or contact form — that users see and interact with in their browsers. The Svelte compiler converts your components to JavaScript that can be run to render the HTML for the page and to CSS that styles the page. You don't need to know Svelte to understand the rest of this guide, but it will help. If you'd like to learn more, check out [the Svelte tutorial](/tutorial).

            ## SvelteKit vs Svelte

            Svelte renders UI components. You can compose these components and render an entire page with just Svelte, but you need more than just Svelte to write an entire app.

            SvelteKit helps you build web apps while following modern best practices and providing solutions to common development challenges. It offers everything from basic functionalities — like a [router](glossary#Routing) that updates your UI when a link is clicked — to more advanced capabilities. Its extensive list of features includes [build optimizations](https://vitejs.dev/guide/features.html#build-optimizations) to load only the minimal required code; [offline support](service-workers); [preloading](link-options#data-sveltekit-preload-data) pages before user navigation; [configurable rendering](page-options) to handle different parts of your app on the server via [SSR](glossary#SSR), in the browser through [client-side rendering](glossary#CSR), or at build-time with [prerendering](glossary#Prerendering); [image optimization](images); and much more. Building an app with all the modern best practices is fiendishly complicated, but SvelteKit does all the boring stuff for you so that you can get on with the creative part.

            It reflects changes to your code in the browser instantly to provide a lightning-fast and feature-rich development experience by leveraging [Vite](https://vitejs.dev/) with a [Svelte plugin](https://github.com/sveltejs/vite-plugin-svelte) to do [Hot Module Replacement (HMR)](https://github.com/sveltejs/vite-plugin-svelte/blob/main/docs/config.md#hot).
        """,
        metadata={"source": "llms.txt"},
    ),
]

#Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
chunks

/home/kevadamar/Projects/Research/ragl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'llms.txt'}, page_content="## Before we begin\n\n            > [!NOTE] If you're new to Svelte or SvelteKit we recommend checking out the [interactive tutorial](/tutorial/kit).\n            >\n            > If you get stuck, reach out for help in the [Discord chatroom](/chat).\n\n            ## What is SvelteKit?"),
 Document(metadata={'source': 'llms.txt'}, page_content="## What is SvelteKit?\n\n            SvelteKit is a framework for rapidly developing robust, performant web applications using [Svelte](../svelte). If you're coming from React, SvelteKit is similar to Next. If you're coming from Vue, SvelteKit is similar to Nuxt.\n\n            To learn more about the kinds of applications you can build with SvelteKit, see the [documentation regarding project types](project-types).\n\n            ## What is Svelte?"),
 Document(metadata={'source': 'llms.txt'}, page_content="In short, Svelte is a way of writing user interface components — like a navigation

# Generate embeddings

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import json

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# Embed all chunks
embeddings = model.encode([chunk.page_content for chunk in chunks])

# Create FAISS index
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# save to json file, for retrieval later
faiss.write_index(index, "rag_index.faiss")
with open("rag_chunks.json","w") as f:
    json.dump([c.model_dump() for c in chunks], f, ensure_ascii=False, indent=2)

In [15]:
# Accept user query and Retrieve context

def retrieve_relevant_chunks(query, top_k=3):
    index = faiss.read_index("rag_index.faiss")
    with open("rag_chunks.json", "r") as f:
        chunks = json.load(f)

    query_vec = model.encode([query])
    distances, indices = index.search(query_vec, top_k)

    return [chunks[i] for i in indices[0]]

# Ask LLM

In [28]:
import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}

Answer the question based on the above context: 
{question}.

Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
Do not use words like "As an AI language model".
If the answer is not contained within the text below, say "I don't know".
"""

query = input("Enter your question: ")
context_text = retrieve_relevant_chunks(query)
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)

response = gemini_model.generate_content(prompt)
answer = response.candidates[0].content.parts[0].text
print("\n💡 Answer:\n", answer)


💡 Answer:
 I don't know.


In [22]:
genai.list_models()

<generator object list_models at 0x7f479c5173d0>